# 从零实现 Double DQN：Q 网络、经验回放与目标网络

本 Notebook 用 PyTorch 手写 `QNetwork.forward`，并搭建一个完整但很小的强化学习闭环：环境交互、epsilon-greedy、replay buffer、Double DQN target、Huber loss、target network 同步、离线评估和可信策略制品。

重点不是“跑出一个高 reward”，而是验证 terminal mask、target detach、online/target 网络职责、随机性和服务边界。受控链式环境的成功不能外推 Atari、机器人或线上推荐。

## 1. 可复现运行合同

环境、探索和 replay sampling 分别使用显式随机源。全程 CPU 单线程。训练日志记录 episode return，但模型选择不读取最终测试种子。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)

from collections import deque
from copy import deepcopy
from dataclasses import dataclass
from hashlib import sha256
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 4101
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
assert DEVICE.type == "cpu"
print({"torch": torch.__version__, "seed": SEED})

## 2. 环境：可手算的 chain MDP

状态是 0–5 的位置，动作 0/1 表示左/右。到达位置 5 得 `+1` 并真正 terminated；其他步 `-0.02`。达到最大步数只是 truncated，Bellman target 是否 bootstrap 应由任务语义决定，不能把 terminated 与 time-limit truncation 混为一谈。

网络输入使用 one-hot，避免把位置编号错误解释为连续尺度。

In [ ]:
class ChainEnv:
    def __init__(self, size=6, max_steps=12):
        if size < 3 or max_steps < size - 1:
            raise ValueError("invalid_chain_config")
        self.size, self.max_steps = size, max_steps
        self.reset()
    def observation(self):
        return F.one_hot(torch.tensor(self.position), self.size).float()
    def reset(self, start=0):
        if not 0 <= start < self.size - 1:
            raise ValueError("invalid_start")
        self.position, self.steps = int(start), 0
        return self.observation()
    def step(self, action):
        if action not in (0, 1):
            raise ValueError("invalid_action")
        self.position = max(0, self.position - 1) if action == 0 else min(self.size - 1, self.position + 1)
        self.steps += 1
        terminated = self.position == self.size - 1
        truncated = self.steps >= self.max_steps and not terminated
        reward = 1.0 if terminated else -0.02
        return self.observation(), reward, terminated, truncated

env_probe = ChainEnv()
state = env_probe.reset()
for _ in range(5): state, reward, terminated, truncated = env_probe.step(1)
assert terminated and not truncated and reward == 1.0
assert int(state.argmax()) == 5
assert ChainEnv(max_steps=5).max_steps == 5

## 3. QNetwork 与动作合同

`forward([B,state_dim]) -> [B,action_dim]` 输出每个动作的未归一化 Q 值，不做 softmax。Q 值是折扣回报估计，不是动作概率。服务时 argmax；训练探索只发生在 agent 边界。

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=32):
        super().__init__()
        self.state_dim, self.action_dim, self.hidden_dim = state_dim, action_dim, hidden_dim
        self.network = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.ReLU(),
                                     nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                     nn.Linear(hidden_dim, action_dim))
    def forward(self, states):
        if states.ndim != 2 or states.shape[1] != self.state_dim:
            raise ValueError("expected_batch_state")
        if not torch.isfinite(states).all():
            raise ValueError("nonfinite_state")
        return self.network(states)

q_probe = QNetwork(6, 2)
assert q_probe(torch.eye(6)).shape == (6, 2)
assert not any(isinstance(layer, nn.Softmax) for layer in q_probe.modules())
try:
    q_probe(torch.zeros(6))
    raise AssertionError("unbatched state must fail")
except ValueError:
    pass

## 4. 经验回放：数据 schema 与随机采样

每条 transition 保存 `state, action, reward, next_state, terminated`。truncated 不写进 terminal mask，本例结束 episode 后仍允许对 time-limit transition bootstrap。生产 replay 还要处理容量、优先级、并发写、策略版本和敏感日志保留。

In [ ]:
@dataclass(frozen=True)
class Transition:
    state: torch.Tensor
    action: int
    reward: float
    next_state: torch.Tensor
    terminated: bool
    truncated: bool

class ReplayBuffer:
    def __init__(self, capacity, seed):
        if capacity <= 0:
            raise ValueError("capacity_must_be_positive")
        self.data = deque(maxlen=capacity)
        self.rng = np.random.default_rng(seed)
    def append(self, transition):
        if transition.action not in (0, 1) or not math.isfinite(transition.reward):
            raise ValueError("invalid_transition")
        if not isinstance(transition.terminated, (bool, np.bool_)) or not isinstance(transition.truncated, (bool, np.bool_)):
            raise TypeError("transition_flags_must_be_bool")
        if transition.state.ndim != 1 or transition.next_state.shape != transition.state.shape:
            raise ValueError("transition_state_shape")
        if not torch.isfinite(transition.state).all() or not torch.isfinite(transition.next_state).all():
            raise ValueError("nonfinite_transition_state")
        self.data.append(transition)
    def sample(self, batch_size):
        if not 1 <= batch_size <= len(self.data):
            raise ValueError("invalid_batch_size")
        indices = self.rng.choice(len(self.data), size=batch_size, replace=False)
        rows = [self.data[int(i)] for i in indices]
        return (torch.stack([x.state for x in rows]),
                torch.tensor([x.action for x in rows], dtype=torch.long),
                torch.tensor([x.reward for x in rows], dtype=torch.float32),
                torch.stack([x.next_state for x in rows]),
                torch.tensor([x.terminated for x in rows], dtype=torch.bool),
                torch.tensor([x.truncated for x in rows], dtype=torch.bool))
    def __len__(self): return len(self.data)

buffer_probe = ReplayBuffer(3, 7)
for i in range(4):
    s = F.one_hot(torch.tensor(i % 3), 6).float()
    buffer_probe.append(Transition(s, i % 2, float(i), s, False, False))
assert len(buffer_probe) == 3
assert buffer_probe.sample(2)[0].shape == (2, 6)
try:
    bad_state41 = torch.zeros(6); bad_state41[0] = float("nan")
    buffer_probe.append(Transition(bad_state41, 0, 0.0, torch.zeros(6), False, False))
    raise AssertionError("NaN replay state must fail")
except ValueError as error:
    assert str(error) == "nonfinite_transition_state"
assert len(buffer_probe) == 3

## 5. Double DQN target 的职责分离

Double DQN 用 online network 选择 `argmax_a Q_online(s',a)`，再由 target network 对该动作估值：

`y = r + gamma * (1-terminated) * Q_target(s', argmax Q_online)`。

target 必须在 `no_grad` 下计算。terminal 样本严格不 bootstrap；普通 DQN 的 `max Q_target` 与 Double DQN 可能选出不同结果。

In [ ]:
@torch.no_grad()
def double_dqn_targets(online, target, next_states, rewards, terminated, gamma):
    if not 0 <= gamma <= 1 or rewards.shape != terminated.shape or rewards.ndim != 1:
        raise ValueError("invalid_target_contract")
    if terminated.dtype != torch.bool or not rewards.is_floating_point():
        raise TypeError("target_flags_or_rewards_dtype")
    if next_states.ndim != 2 or next_states.shape[0] != len(rewards):
        raise ValueError("target_batch_mismatch")
    if not torch.isfinite(next_states).all() or not torch.isfinite(rewards).all():
        raise ValueError("nonfinite_target_input")
    next_actions = online(next_states).argmax(dim=1)
    next_values = target(next_states).gather(1, next_actions[:, None]).squeeze(1)
    return rewards + gamma * (~terminated).float() * next_values

class FixedQ(nn.Module):
    def __init__(self, values):
        super().__init__(); self.register_buffer("values", torch.tensor(values, dtype=torch.float32))
    def forward(self, states): return self.values[: len(states)]

online_fixed = FixedQ([[1,3], [5,2]])
target_fixed = FixedQ([[10,7], [4,9]])
oracle_targets = double_dqn_targets(online_fixed, target_fixed, torch.zeros(2,6),
                                    torch.tensor([0.5,2.0]), torch.tensor([False,True]), 0.9)
assert torch.allclose(oracle_targets, torch.tensor([6.8, 2.0]))
assert not oracle_targets.requires_grad
plain_dqn_first = 0.5 + 0.9 * 10.0
assert not math.isclose(float(oracle_targets[0]), plain_dqn_first)
try:
    double_dqn_targets(online_fixed, target_fixed, torch.zeros(2,6), torch.tensor([0.5,2.0]), torch.tensor([0,1]), 0.9)
    raise AssertionError("integer terminated mask must fail")
except TypeError as error:
    assert str(error) == "target_flags_or_rewards_dtype"
assert oracle_targets.shape == (2,)

## 6. 训练循环：探索、replay 与 target 同步

前若干 transition 先填充 replay。epsilon 逐步下降但保留探索下限；每次更新只对 chosen action 的 Q 计算 Huber loss。target network 定期硬同步且从不被 optimizer 更新。

下面固定 episode 数，不根据 test return 调参。真实项目应保存环境版本、reward 定义、行为策略和评估种子。

In [ ]:
def choose_action(network, state, epsilon, rng):
    if not 0 <= epsilon <= 1:
        raise ValueError("epsilon_out_of_range")
    if rng.random() < epsilon:
        return int(rng.integers(0, network.action_dim))
    with torch.no_grad(): return int(network(state[None]).argmax(dim=1).item())

torch.manual_seed(SEED + 1)
online41, target41 = QNetwork(6,2), QNetwork(6,2)
target41.load_state_dict(online41.state_dict()); target41.requires_grad_(False); target41.eval()
optimizer41 = torch.optim.Adam(online41.parameters(), lr=0.006)
replay41 = ReplayBuffer(500, SEED + 2)
env41 = ChainEnv()
explore_rng = np.random.default_rng(SEED + 3)
TARGET_SYNC_INTERVAL41 = 25
BOOTSTRAP_ON_TRUNCATION41 = True
returns41, losses41 = [], []
initial_online41 = {k: v.detach().clone() for k,v in online41.state_dict().items()}
updates41 = 0
for episode in range(120):
    state = env41.reset(start=int(explore_rng.integers(0, 3)))
    episode_return = 0.0
    epsilon = max(0.05, 0.9 - episode / 100)
    for _ in range(env41.max_steps):
        action = choose_action(online41, state, epsilon, explore_rng)
        next_state, reward, terminated, truncated = env41.step(action)
        replay41.append(Transition(state.clone(), action, reward, next_state.clone(), terminated, truncated))
        state, episode_return = next_state, episode_return + reward
        if len(replay41) >= 32:
            states, actions, rewards, next_states, terminals, truncations = replay41.sample(32)
            assert truncations.dtype == torch.bool
            predicted = online41(states).gather(1, actions[:,None]).squeeze(1)
            targets = double_dqn_targets(online41, target41, next_states, rewards, terminals, gamma=0.95)
            loss = F.smooth_l1_loss(predicted, targets)
            optimizer41.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(online41.parameters(), 5.0); optimizer41.step()
            losses41.append(float(loss.detach())); updates41 += 1
            if updates41 % TARGET_SYNC_INTERVAL41 == 0: target41.load_state_dict(online41.state_dict())
        if terminated or truncated: break
    returns41.append(episode_return)

target41.load_state_dict(online41.state_dict()); target41.eval(); online41.eval()
changed41 = any(not torch.equal(initial_online41[k], v) for k,v in online41.state_dict().items())
assert changed41 and updates41 > 100
assert np.mean(returns41[-30:]) > np.mean(returns41[:30])
assert losses41 and np.isfinite(losses41).all()
print({"mean_return_first30": np.mean(returns41[:30]), "last30": np.mean(returns41[-30:]), "updates": updates41})

## 7. 独立 greedy 评估与 Q 值诊断

评估 epsilon 固定为 0，不写 replay、不更新权重。除了 success rate，还检查从每个非终止位置的 greedy action 和 Q 值有限性。链式环境的最优动作始终向右，是一个可解释 oracle。

In [ ]:
@torch.no_grad()
def evaluate_policy(network, starts):
    successes, lengths = [], []
    before = {k: v.clone() for k,v in network.state_dict().items()}
    for start in starts:
        env = ChainEnv(); state = env.reset(start)
        for step in range(env.max_steps):
            action = int(network(state[None]).argmax(1).item())
            state, _, terminated, truncated = env.step(action)
            if terminated or truncated: break
        successes.append(terminated); lengths.append(step + 1)
    assert all(torch.equal(before[k], v) for k,v in network.state_dict().items())
    return float(np.mean(successes)), lengths

success_rate41, eval_lengths41 = evaluate_policy(online41, [0,1,2,3,4] * 5)
with torch.no_grad():
    all_q41 = online41(torch.eye(6)[:5]); greedy_actions41 = all_q41.argmax(1)
print({"success_rate": success_rate41, "greedy_actions": greedy_actions41.tolist(), "lengths": eval_lengths41[:5]})
assert success_rate41 == 1.0
assert torch.equal(greedy_actions41, torch.ones(5, dtype=torch.long))
assert torch.isfinite(all_q41).all()

## 8. 策略制品、动作语义与在线边界

仅保存 state_dict 不够：动作 `0/1` 的语义、state encoder、环境/reward 版本、gamma 与网络 config 都必须绑定。公开接口只按受信 policy version 加载模型，并在每次决策前校验；客户端不能提交任意 `nn.Module`。

真实线上 RL 还需要离线安全评估、行为策略覆盖、约束动作过滤、熔断与人工接管。hash 只做一致性检查，发布仍需签名。

In [ ]:
def state_hash41(module):
    digest = sha256()
    for name,value in sorted(module.state_dict().items()):
        array=value.detach().cpu().contiguous().numpy()
        digest.update(name.encode()); digest.update(str(array.dtype).encode())
        digest.update(json.dumps(list(array.shape)).encode()); digest.update(array.tobytes())
    return digest.hexdigest()

def config41(model): return {"state_dim": model.state_dim, "action_dim": model.action_dim, "hidden_dim": model.hidden_dim}
def bundle_hash41(value):
    return sha256(json.dumps({k:v for k,v in value.items() if k!="bundle_sha256"},sort_keys=True).encode()).hexdigest()

artifact41 = {"policy_version":"double-dqn-chain-v1", "architecture":"QNetwork",
              "config":config41(online41), "state_sha256":state_hash41(online41),
              "state_encoder":"chain-one-hot-v1", "action_map":{"0":"left","1":"right"},
              "environment_version":"chain-6-v1", "reward_version":"goal1-step-0.02-v1", "gamma":0.95,
              "training_semantics":{"bootstrap_on_truncation":BOOTSTRAP_ON_TRUNCATION41,
                                    "target_sync_interval":TARGET_SYNC_INTERVAL41,
                                    "replay_capacity":500,"training_episodes":120,"seed":SEED}}
artifact41["bundle_sha256"] = bundle_hash41(artifact41)
_REGISTRY41 = {artifact41["policy_version"]:{"model":online41,"artifact":deepcopy(artifact41)}}

def load_policy41(version):
    if version not in _REGISTRY41: raise KeyError("unknown_policy")
    entry=_REGISTRY41[version]; model,artifact=entry["model"],entry["artifact"]
    if type(model) is not QNetwork or artifact["architecture"] != type(model).__name__: raise TypeError("architecture_mismatch")
    if artifact["bundle_sha256"] != bundle_hash41(artifact): raise RuntimeError("bundle_mismatch")
    if artifact["config"] != config41(model) or artifact["state_sha256"] != state_hash41(model): raise RuntimeError("model_mismatch")
    semantics = artifact.get("training_semantics", {})
    if semantics.get("bootstrap_on_truncation") is not True or semantics.get("target_sync_interval") != TARGET_SYNC_INTERVAL41:
        raise RuntimeError("training_semantics_mismatch")
    return model,artifact

@torch.no_grad()
def policy_action41(state, version="double-dqn-chain-v1"):
    model,artifact=load_policy41(version)
    raw=torch.as_tensor(state,dtype=torch.float32)
    if raw.shape != (artifact["config"]["state_dim"],) or not torch.isfinite(raw).all(): raise ValueError("invalid_online_state")
    if not torch.allclose(raw.sum(),torch.tensor(1.0)) or not bool(((raw==0)|(raw==1)).all()): raise ValueError("state_encoder_mismatch")
    action=int(model(raw[None]).argmax(1).item())
    return action,{"policy_version":version,"action_name":artifact["action_map"][str(action)],"bundle_sha256":artifact["bundle_sha256"]}

served_action41, trace41 = policy_action41(torch.eye(6)[0])
assert served_action41 == 1 and trace41["action_name"] == "right"
parameter41=next(online41.parameters()); backup41=parameter41.detach().clone()
try:
    with torch.no_grad(): parameter41.add_(0.25)
    try: policy_action41(torch.eye(6)[0]); raise AssertionError("tampered policy must fail")
    except RuntimeError as error: assert str(error)=="model_mismatch"
finally:
    with torch.no_grad(): parameter41.copy_(backup41)
action_map_backup41 = deepcopy(_REGISTRY41["double-dqn-chain-v1"]["artifact"]["action_map"])
try:
    _REGISTRY41["double-dqn-chain-v1"]["artifact"]["action_map"]["1"] = "unsafe-remap"
    try: load_policy41("double-dqn-chain-v1"); raise AssertionError("tampered action contract must fail")
    except RuntimeError as error: assert str(error)=="bundle_mismatch"
finally:
    _REGISTRY41["double-dqn-chain-v1"]["artifact"]["action_map"] = action_map_backup41
try: policy_action41(torch.zeros(6)); raise AssertionError("invalid one-hot must fail")
except ValueError as error: assert str(error)=="state_encoder_mismatch"
assert served_action41 in (0, 1)

## 9. 面试总结与来源

DQN 的核心不是 MLP，而是用 replay 降低样本相关性、用 target network 稳定 bootstrap。Double DQN 把动作选择与估值拆开：`online(next_state).argmax` 负责选动作，`target(next_state)` 只对该动作估值，从而缓解同一组噪声同时参与选择和估值造成的过估计。

目标值是 `r + gamma * (1-terminated) * Q_target(s', argmax_a Q_online(s',a))`。时间上限导致的 `truncated` 是否继续 bootstrap 是任务语义，不能偷懒并入 terminal；本例把该选择连同 target 同步周期、replay 容量和随机种子写进发布合同。上线还要检查离线数据 coverage、行为策略偏差、动作约束、回滚阈值与 shadow/canary 指标。

- Mnih et al., *Human-level control through deep reinforcement learning*：https://www.nature.com/articles/nature14236
- van Hasselt et al., *Deep Reinforcement Learning with Double Q-learning*：https://arxiv.org/abs/1509.06461
- PyTorch autograd notes：https://pytorch.org/docs/stable/notes/autograd.html

这里没有 prioritized replay、n-step return、distributional Q、连续动作或真实离线策略评估。

In [ ]:
assert type(online41).__name__ == "QNetwork"
assert not any(p.requires_grad for p in target41.parameters())
assert optimizer41.param_groups[0]["params"][0] is next(online41.parameters())
assert success_rate41 == 1.0
assert all(action == 1 for action in greedy_actions41.tolist())
assert state_hash41(online41) == artifact41["state_sha256"]
assert bundle_hash41(artifact41) == artifact41["bundle_sha256"]
assert policy_action41(torch.eye(6)[4])[0] == 1
print("Double DQN、终止语义、评估与可信策略回归全部通过。")